# PCB test-point placement — DreamerV3 cold-start training (A100, 20 traces)

Runs the full cold-start training pipeline (expert demos + decayed behavior cloning + potential reward shaping) on the canonical 20-trace board, then scores the trained policy against the classical baselines. Everything — demos, replay, checkpoints, TensorBoard logs — is saved to your Google Drive as it runs.

**Setup:** `Runtime → Change runtime type → A100 GPU` (Colab Pro), then run all cells top to bottom. The Drive cell asks for authorization once.

Timeline: demo generation ~10–15 min (one-time, cached in Drive), then 100k training steps ≈ 3–6 h with checkpoints every 5k steps. Interrupt — or lose the runtime — anytime: re-running the notebook resumes exactly where it left off. On a free T4 instead, set `CONFIG = "colab"` and `NUM_TRACES = 8` in the settings cell (~2–4 h for 30k steps).

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100 GPU"
name = torch.cuda.get_device_name(0)
print("torch", torch.__version__, "|", name)
if "A100" not in name:
    print("NOTE: not an A100 — everything still runs, just slower; "
          "consider CONFIG='colab', NUM_TRACES=8 in the settings cell.")

In [ ]:
# All logs, demos, and checkpoints live in your Google Drive, so nothing is
# lost on disconnect and re-running this notebook later resumes training.
from google.colab import drive
drive.mount("/content/drive")
LOGROOT = "/content/drive/MyDrive/pcb-router-logs"
print("backing up to:", LOGROOT)

In [ ]:
# Get the code from GitHub main (pulls the latest on re-runs).
import pathlib
if not pathlib.Path("/content/pcb-router").exists():
    !git clone -q https://github.com/pauljiang03/pcb-router /content/pcb-router
%cd /content/pcb-router
!git pull --ff-only

In [ ]:
# Dependencies (torch/numpy/tensorboard/matplotlib ship with Colab) and a
# quick sanity run of the cold-start tests (~10 s).
%pip -q install gymnasium "ruamel.yaml" openpyxl
!python -m pytest tests/test_coldstart.py -q

In [ ]:
NUM_TRACES = 20        # canonical board size (T4 fallback: 8)
CONFIG = "colab_a100"  # configs.yaml section: 100k steps, eval every 5k (T4 fallback: "colab")
# 4 env workers overlap CPU routing with GPU training (A100 VMs have 12 vCPUs).
# Set to "" if the worker processes misbehave.
ENV_FLAGS = "--envs 4 --parallel"
COLD_DIR = f"{LOGROOT}/cold"
print(NUM_TRACES, "traces |", CONFIG, "|", COLD_DIR)

In [ ]:
# Live training curves. Key scalars: eval_return / train_return (totals are
# unchanged by the shaping, so they read like the legacy reward), log_routed
# (nets routed, target = NUM_TRACES), log_layers (target 1), log_phi
# (placement potential), bc_loss + bc_scale (imitation term; decays to 0
# over bc_decay steps — 40k on the A100 config).
%load_ext tensorboard
import tensorboard.notebook as tbnb
tbnb.start("--logdir " + LOGROOT)

## Train — cold start (demos + BC + shaping)

First run generates 200 expert episodes into `demo_eps/` in your Drive using 8 parallel CPU workers (~10–20 min at 20 traces; hard moat boards cost ~30s each, so serial would take over an hour). Progress prints as it goes. Then it trains 100k steps. Interrupt anytime; re-run this cell to resume — finished demos are never regenerated.

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{COLD_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS}

In [ ]:
# Score the trained policy against the classical baselines on the SAME boards
# (the summary table at the bottom is the headline result: compare the
# Dreamer row to Smart on failures / max / spread). Add --board_seed 1000000
# to score on held-out boards instead of the fixed TE board; add --fast for
# a quick low-budget pass (the Random baseline costs ~30s/episode at the
# quality budget on 20-trace boards).
!python eval.py --checkpoint "{COLD_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot

## Reading the results

- **TensorBoard `eval_return`** — should jump to roughly demo level within the first few thousand steps (that's BC pulling the policy to the expert), hold through the decay window, then grind upward as RL refines. A flat curve at a large negative value is the no-learning plateau — if you see that, check that `bc_loss` exists and `log_routed` is near 20.
- **`log_routed` / `log_layers`** — should sit near 20 and 1 almost immediately (the demos are planar single-layer placements).
- **`log_max_len` / `log_spread`** — once `bc_scale` decays to 0 (40k steps), these grinding down IS the remaining objective: shorter, more equal traces.
- **`eval.py` summary table** — the scoreboard vs. classical baselines on identical boards. Matching Smart on the TE board is success (it is near-optimal there); the learned policy's edge should show on held-out boards (`--board_seed 1000000`) and in inference speed. Watch that `log_routed` never dips while return improves — that would mean the length pressure is beating the failure penalty.

Everything is already backed up in your Drive under `pcb-router-logs/` — stopping the runtime loses nothing.

In [ ]:
# Optional: download a local copy of the checkpoint + TensorBoard events.
# (Your Drive already has everything under pcb-router-logs/.)
import pathlib, shutil
out = pathlib.Path("/content/results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
d = pathlib.Path(COLD_DIR)
for f in list(d.glob("events*")) + [d / "latest.pt"]:
    if f.exists():
        shutil.copy(f, out / f.name)
shutil.make_archive("/content/pcb_router_results", "zip", out)
from google.colab import files
files.download("/content/pcb_router_results.zip")